# Лабораторная работа №4. Шифрование с открытым ключом
## Вариант 7 — Многократное шифрование Merkle-Hellman (сложный уровень)

**Цель работы:** изучение методов шифрования с открытым ключом (асимметричного шифрования) на примере многократной схемы Merkle-Hellman (рюкзачная криптосистема).

**План работы:**
1. Теоретические сведения и алгоритм.
2. Вспомогательные алгоритмы из Приложения (расширенный алгоритм Евклида 2.107, решение задачи сверхвозрастающего подмножества сумм).
3. Генерация ключа, шифрование, дешифрование.
4. Проверка на примере из методички (базовая схема, t = 1).
5. Пошаговый пример многократной схемы (t = 3).
6. Шифрование и дешифрование текстового сообщения.
7. Автоматические тесты.
8. Сохранение ключей в файлы, шифрование и дешифрование текстовых файлов.
9. Тесты работы с файлами, выводы.

## 1. Теория

**Сверхвозрастающая последовательность** — последовательность положительных целых $(b_1, \dots, b_n)$, для которой
$b_i > \sum_{j=1}^{i-1} b_j,\ 2 \le i \le n.$

**Решение задачи сверхвозрастающего подмножества сумм.** По $(b_1,\dots,b_n)$ и числу $s$, являющемуся суммой подмножества $b_i$, находим $(x_1,\dots,x_n)$, $x_i \in \{0,1\}$, $\sum x_i b_i = s$: идём от $i=n$ до $1$; если $s \ge b_i$, то $x_i = 1,\ s \leftarrow s - b_i$, иначе $x_i = 0$.

### Генерация ключа (многократная схема)
1. $n$ и $t$ — фиксированные параметры системы.
2. Выбрать сверхвозрастающую последовательность $(a_1^{(0)}, \dots, a_n^{(0)})$.
3. Для $j = 1, \dots, t$:
   * выбрать модуль $M_j > a_1^{(j-1)} + \dots + a_n^{(j-1)}$;
   * выбрать $W_j$, $1 \le W_j \le M_j - 1$, $\gcd(W_j, M_j) = 1$;
   * вычислить $a_i^{(j)} = a_i^{(j-1)} W_j \bmod M_j,\ i = 1..n$.
4. Выбрать случайную перестановку $\pi$ множества $\{1,\dots,n\}$.
5. **Открытый ключ:** $(a_1,\dots,a_n)$, где $a_i = a^{(t)}_{\pi(i)}$.
   **Закрытый ключ:** $(\pi, M_1,\dots,M_t, W_1,\dots,W_t, a_1^{(0)},\dots,a_n^{(0)})$.

### Шифрование
Сообщение — бинарная строка $m = m_1 \dots m_n$; $c = \sum_{i=1}^n m_i a_i$.

### Дешифрование
1. $d_{t+1} = c$; для $j = t, t-1, \dots, 1$: $d_j = W_j^{-1} d_{j+1} \bmod M_j$.
2. Решить задачу сверхвозрастающего подмножества сумм $d_1 = r_1 a_1^{(0)} + \dots + r_n a_n^{(0)}$.
3. $m_i = r_{\pi(i)},\ i = 1..n$.

При $t = 1$ получается базовая схема Merkle-Hellman.

## 2. Вспомогательные алгоритмы

In [8]:
import random
from math import gcd


def extended_euclid(a, b):
    """Алгоритм 2.107 (расширенный алгоритм Евклида), a >= b >= 0.
    Возвращает (d, x, y): d = gcd(a, b), a*x + b*y = d."""
    if b == 0:
        return a, 1, 0
    x2, x1, y2, y1 = 1, 0, 0, 1
    while b > 0:
        q = a // b
        r = a - q * b
        x = x2 - q * x1
        y = y2 - q * y1
        a, b = b, r
        x2, x1 = x1, x
        y2, y1 = y1, y
    return a, x2, y2


def mod_inverse(w, m):
    """W^{-1} mod M через расширенный алгоритм Евклида (требуется gcd(W, M) = 1)."""
    d, _, y = extended_euclid(m, w % m)      # m*x + w*y = d
    if d != 1:
        raise ValueError("Обратного элемента не существует: gcd(W, M) != 1")
    return y % m


def is_superincreasing(seq):
    """Проверка свойства b_i > b_1 + ... + b_{i-1}."""
    total = 0
    for b in seq:
        if b <= total:
            return False
        total += b
    return True


def solve_superincreasing(seq, s):
    """Решение задачи сверхвозрастающего подмножества сумм (алгоритм из методички)."""
    n = len(seq)
    x = [0] * n
    i = n - 1
    while i >= 0:
        if s >= seq[i]:
            x[i] = 1
            s -= seq[i]
        i -= 1
    if s != 0:
        raise ValueError("Число s не является суммой подмножества последовательности")
    return x

## 3. Генерация ключа, шифрование и дешифрование

In [9]:
def generate_superincreasing(n, rng=random, step_bits=8):
    """Случайная сверхвозрастающая последовательность длины n."""
    seq, total = [], 0
    for _ in range(n):
        b = total + rng.randint(1, 2 ** step_bits)
        seq.append(b)
        total += b
    return seq


def build_public_key(a0, Ms, Ws, perm):
    """По параметрам закрытого ключа строит открытый ключ.
    perm — список, perm[i-1] = pi(i) (нумерация с 1).
    Возвращает (public_key, levels), levels[j] = a^{(j)}."""
    n = len(a0)
    assert is_superincreasing(a0), "a^(0) должна быть сверхвозрастающей"
    assert sorted(perm) == list(range(1, n + 1)), "perm — перестановка {1..n}"
    assert len(Ms) == len(Ws)
    cur = list(a0)
    levels = [cur]
    for M, W in zip(Ms, Ws):
        assert M > sum(cur), "требуется M_j > сумма a^(j-1)"
        assert 1 <= W <= M - 1 and gcd(W, M) == 1, "требуется 1 <= W_j <= M_j-1, НОД(W_j, M_j)=1"
        cur = [(x * W) % M for x in cur]
        levels.append(cur)
    public = [cur[perm[i] - 1] for i in range(n)]      # a_i = a^{(t)}_{pi(i)}
    return public, levels


def generate_keys(n, t, rng=random, step_bits=8):
    """Генерация ключа многократной схемы Merkle-Hellman."""
    a0 = generate_superincreasing(n, rng, step_bits)
    cur = list(a0)
    Ms, Ws = [], []
    for _ in range(t):
        M = sum(cur) + rng.randint(1, 2 ** step_bits)       # M_j > sum a^(j-1)
        while True:
            W = rng.randint(1, M - 1)
            if gcd(W, M) == 1:
                break
        cur = [(x * W) % M for x in cur]
        Ms.append(M)
        Ws.append(W)
    perm = list(range(1, n + 1))
    rng.shuffle(perm)
    public, levels = build_public_key(a0, Ms, Ws, perm)
    private = {"perm": perm, "Ms": Ms, "Ws": Ws, "a0": a0}
    return public, private


def encrypt(bits, public):
    """c = m_1 a_1 + ... + m_n a_n; bits — список из n бит."""
    if len(bits) != len(public):
        raise ValueError("Длина сообщения должна быть равна n")
    return sum(m * a for m, a in zip(bits, public))


def decrypt(c, private, verbose=False):
    """Дешифрование: d_{t+1}=c, d_j = W_j^{-1} d_{j+1} mod M_j, затем решение
    сверхвозрастающей задачи и перестановка m_i = r_{pi(i)}."""
    perm, Ms, Ws, a0 = private["perm"], private["Ms"], private["Ws"], private["a0"]
    t = len(Ms)
    d = c
    for j in range(t, 0, -1):
        d = (mod_inverse(Ws[j - 1], Ms[j - 1]) * d) % Ms[j - 1]
        if verbose:
            print(f"d_{j} = W_{j}^-1 * d_{j+1} mod M_{j} = {d}")
    r = solve_superincreasing(a0, d)
    if verbose:
        print("r =", r)
    return [r[perm[i] - 1] for i in range(len(a0))]

## 4. Проверка на примере из методички (базовая схема, $t = 1$)

Параметры: $n=6$, $(12,17,33,74,157,316)$, $M=737$, $W=635$, $\pi(1..6)=3,6,1,2,5,4$, $m = 101101$.
Ожидается: открытый ключ $(319,196,250,477,200,559)$, $c = 1605$, $d = 136$.

In [10]:
a0 = [12, 17, 33, 74, 157, 316]
M, W = 737, 635
perm = [3, 6, 1, 2, 5, 4]

public, levels = build_public_key(a0, [M], [W], perm)
private = {"perm": perm, "Ms": [M], "Ws": [W], "a0": a0}
print("Открытый ключ:", public)
assert public == [319, 196, 250, 477, 200, 559]

m = [1, 0, 1, 1, 0, 1]
c = encrypt(m, public)
print("c =", c)
assert c == 1605

print("W^-1 mod M =", mod_inverse(W, M))
m_dec = decrypt(c, private, verbose=True)
print("Расшифровано:", m_dec)
assert m_dec == m
print("Пример из методички воспроизведён верно.")

Открытый ключ: [319, 196, 250, 477, 200, 559]
c = 1605
W^-1 mod M = 513
d_1 = W_1^-1 * d_2 mod M_1 = 136
r = [1, 1, 1, 1, 0, 0]
Расшифровано: [1, 0, 1, 1, 0, 1]
Пример из методички воспроизведён верно.


## 5. Пошаговый пример многократной схемы ($n = 8$, $t = 3$)

In [11]:
rng = random.Random(7)          # фиксированное зерно для воспроизводимости
n, t = 8, 3
public, private = generate_keys(n, t, rng)

print("Сверхвозрастающая a^(0):", private["a0"])
for j in range(t):
    print(f"M_{j+1} = {private['Ms'][j]},  W_{j+1} = {private['Ws'][j]}")
print("Перестановка pi(1..n):", private["perm"])
_, levels = build_public_key(private["a0"], private["Ms"], private["Ws"], private["perm"])
for j, lv in enumerate(levels):
    print(f"a^({j}) =", lv)
print("Открытый ключ (a_1..a_n):", public)

m = [1, 0, 1, 1, 0, 0, 1, 0]
print("\nСообщение m =", m)
c = encrypt(m, public)
print("Шифротекст c =", c)

print("\nДешифрование:")
m_dec = decrypt(c, private, verbose=True)
print("Восстановлено:", m_dec)
assert m_dec == m

Сверхвозрастающая a^(0): [166, 244, 613, 1048, 2109, 4229, 8597, 17036]
M_1 = 34152,  W_1 = 5633
M_2 = 166089,  W_2 = 109622
M_3 = 965947,  W_3 = 252354
Перестановка pi(1..n): [8, 3, 6, 7, 1, 4, 5, 2]
a^(0) = [166, 244, 613, 1048, 2109, 4229, 8597, 17036]
a^(1) = [12974, 8372, 3677, 29240, 29253, 18013, 33517, 30820]
a^(2) = [15721, 113659, 148180, 161758, 92043, 155054, 145805, 133691]
a^(3) = [112905, 439015, 75456, 324059, 257660, 881987, 587793, 793692]
Открытый ключ (a_1..a_n): [793692, 75456, 881987, 587793, 112905, 324059, 257660, 439015]

Сообщение m = [1, 0, 1, 1, 0, 0, 1, 0]
Шифротекст c = 2521132

Дешифрование:
d_3 = W_3^-1 * d_4 mod M_3 = 526593
d_2 = W_2^-1 * d_3 mod M_2 = 111603
d_1 = W_1^-1 * d_2 mod M_1 = 31971
r = [0, 0, 0, 0, 1, 1, 1, 1]
Восстановлено: [1, 0, 1, 1, 0, 0, 1, 0]


## 6. Шифрование текстового сообщения

In [12]:
def text_to_blocks(text, n):
    """Текст -> UTF-8 -> битовый поток -> блоки по n бит (последний дополняется нулями)."""
    data = text.encode("utf-8")
    bits = [int(b) for byte in data for b in format(byte, "08b")]
    while len(bits) % n:
        bits.append(0)
    return [bits[i:i + n] for i in range(0, len(bits), n)], len(data)


def blocks_to_text(blocks, byte_len):
    bits = [b for block in blocks for b in block]
    data = bytes(int("".join(map(str, bits[i:i + 8])), 2) for i in range(0, byte_len * 8, 8))
    return data.decode("utf-8")


rng = random.Random(2026)
n, t = 16, 4
public, private = generate_keys(n, t, rng)
print(f"n = {n}, t = {t}")
print("Модули M_j:", private["Ms"])

message = "Лабораторная работа №4: Merkle-Hellman"
blocks, byte_len = text_to_blocks(message, n)
cipher = [encrypt(b, public) for b in blocks]
print("\nИсходный текст:", message)
print("Шифротекст (числа):", cipher)

decoded = blocks_to_text([decrypt(c, private) for c in cipher], byte_len)
print("Расшифрованный текст:", decoded)
assert decoded == message

n = 16, t = 4
Модули M_j: [6447549, 50258666, 330175592, 2913919185]

Исходный текст: Лабораторная работа №4: Merkle-Hellman
Шифротекст (числа): [12435826338, 8817892315, 9337633384, 15947543361, 7170412300, 8817892315, 10015541234, 15947543361, 7170412300, 13622155496, 8817892315, 14819804415, 3053774794, 4329873258, 5812241994, 6656152012, 11240496649, 6721354752, 5820350937, 11251380653, 12079224064, 6504196146, 11339060106, 14318688298, 11337300151, 6795704326, 12684172414, 13290356776, 13942622567]
Расшифрованный текст: Лабораторная работа №4: Merkle-Hellman


## 7. Автоматические тесты

In [13]:
rng = random.Random(12345)
total = 0
for n in (4, 8, 16, 32):
    for t in (1, 2, 3, 5):
        for _ in range(50):
            pub, priv = generate_keys(n, t, rng)
            assert is_superincreasing(priv["a0"])
            assert all(M > sum(lv) for M, lv in zip(
                priv["Ms"], build_public_key(priv["a0"], priv["Ms"], priv["Ws"], priv["perm"])[1]))
            m = [rng.randint(0, 1) for _ in range(n)]
            assert decrypt(encrypt(m, pub), priv) == m
            total += 1
# крайние сообщения
pub, priv = generate_keys(8, 3, rng)
for m in ([0] * 8, [1] * 8):
    assert decrypt(encrypt(m, pub), priv) == m
# обратные элементы
for _ in range(1000):
    M = rng.randint(3, 10**6)
    W = rng.randint(1, M - 1)
    if gcd(W, M) == 1:
        assert (W * mod_inverse(W, M)) % M == 1
print(f"Все тесты пройдены ({total} случайных ключей/сообщений).")

Все тесты пройдены (800 случайных ключей/сообщений).


## 8. Сохранение ключей в файлы и работа с текстовыми файлами

Ключи сохраняются в JSON-файлы (`public_key.json`, `private_key.json`), шифротекст — в `cipher.json`.
Текст для шифрования читается из файла `input.txt` (если файла нет, создаётся пример; можно положить рядом свой файл). Для дешифрования используются **только** ключ и шифротекст, загруженные из файлов.

In [14]:
import json
import os

WORK_DIR = "lab4_files"
os.makedirs(WORK_DIR, exist_ok=True)

PUB_PATH = os.path.join(WORK_DIR, "public_key.json")
PRIV_PATH = os.path.join(WORK_DIR, "private_key.json")
INPUT_PATH = os.path.join(WORK_DIR, "input.txt")
CIPHER_PATH = os.path.join(WORK_DIR, "cipher.json")
OUTPUT_PATH = os.path.join(WORK_DIR, "decrypted.txt")


def save_keys(public, private, pub_path=PUB_PATH, priv_path=PRIV_PATH):
    """Сохраняет открытый и закрытый ключи в JSON-файлы."""
    with open(pub_path, "w", encoding="utf-8") as f:
        json.dump({"scheme": "Merkle-Hellman (multiple)", "n": len(public), "a": public},
                  f, ensure_ascii=False, indent=2)
    with open(priv_path, "w", encoding="utf-8") as f:
        json.dump({"scheme": "Merkle-Hellman (multiple)", "n": len(private["a0"]),
                   "t": len(private["Ms"]), "perm": private["perm"],
                   "Ms": private["Ms"], "Ws": private["Ws"], "a0": private["a0"]},
                  f, ensure_ascii=False, indent=2)


def load_public_key(path=PUB_PATH):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    public = data["a"]
    assert data["n"] == len(public), "Файл открытого ключа повреждён"
    return public


def load_private_key(path=PRIV_PATH):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    private = {k: data[k] for k in ("perm", "Ms", "Ws", "a0")}
    n, t = data["n"], data["t"]
    assert len(private["a0"]) == n and len(private["perm"]) == n, "Файл закрытого ключа повреждён"
    assert len(private["Ms"]) == t and len(private["Ws"]) == t, "Файл закрытого ключа повреждён"
    assert is_superincreasing(private["a0"]), "a^(0) не сверхвозрастающая"
    return private


def encrypt_file(in_path, out_path, pub_path=PUB_PATH):
    """Читает текстовый файл (UTF-8), шифрует открытым ключом, пишет шифротекст в JSON."""
    public = load_public_key(pub_path)
    with open(in_path, encoding="utf-8", newline="") as f:   # newline="" — не трогаем \r\n
        text = f.read()
    blocks, byte_len = text_to_blocks(text, len(public))
    cipher = [encrypt(b, public) for b in blocks]
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({"n": len(public), "byte_len": byte_len, "cipher": cipher}, f)
    return cipher


def decrypt_file(cipher_path, out_path, priv_path=PRIV_PATH):
    """Читает шифротекст из JSON, расшифровывает закрытым ключом, пишет текст в файл."""
    private = load_private_key(priv_path)
    with open(cipher_path, encoding="utf-8") as f:
        data = json.load(f)
    if data["n"] != len(private["a0"]):
        raise ValueError("Размер блока шифротекста не совпадает с n закрытого ключа")
    blocks = [decrypt(c, private) for c in data["cipher"]]
    text = blocks_to_text(blocks, data["byte_len"])
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        f.write(text)
    return text


# --- исходный текст: берём input.txt, если его нет — создаём пример ---
if not os.path.exists(INPUT_PATH):
    with open(INPUT_PATH, "w", encoding="utf-8", newline="") as f:
        f.write("Лабораторная работа №4. Шифрование с открытым ключом.\n"
                "Схема Merkle-Hellman (многократное шифрование), вариант 7.\n"
                "Symbols: 0123456789 !?@#$%^&*()  Ünïcödé — тест.\n")

# --- 1) генерация и сохранение ключей ---
public, private = generate_keys(n=16, t=3, rng=random.Random(4))
save_keys(public, private)
print("Ключи сохранены:", PUB_PATH, PRIV_PATH)

# --- 2) шифрование файла (нужен только открытый ключ из файла) ---
cipher = encrypt_file(INPUT_PATH, CIPHER_PATH)
print("Шифротекст сохранён:", CIPHER_PATH, f"({len(cipher)} блоков)")

# --- 3) дешифрование файла (нужны только закрытый ключ и шифротекст из файлов) ---
text = decrypt_file(CIPHER_PATH, OUTPUT_PATH)
print("Расшифрованный текст сохранён:", OUTPUT_PATH)
print("-" * 40)
print(text)

with open(INPUT_PATH, encoding="utf-8", newline="") as f:
    original = f.read()
assert text == original
print("Расшифрованный файл совпадает с исходным.")

Ключи сохранены: lab4_files\public_key.json lab4_files\private_key.json
Шифротекст сохранён: lab4_files\cipher.json (126 блоков)
Расшифрованный текст сохранён: lab4_files\decrypted.txt
----------------------------------------
Лабораторная работа №4. Шифрование с открытым ключом.
Схема Merkle-Hellman (многократное шифрование), вариант 7.
Symbols: 0123456789 !?@#$%^&*()  Ünïcödé — тест.

Расшифрованный файл совпадает с исходным.


## 9. Тесты шифрования и дешифрования файлов

In [15]:
import tempfile


def roundtrip(text, n, t, seed):
    """Полный цикл через файлы: ключи -> файл -> шифротекст -> файл -> текст."""
    with tempfile.TemporaryDirectory() as d:
        p = lambda name: os.path.join(d, name)
        pub, priv = generate_keys(n, t, random.Random(seed))
        save_keys(pub, priv, p("pub.json"), p("priv.json"))
        with open(p("in.txt"), "w", encoding="utf-8", newline="") as f:
            f.write(text)
        encrypt_file(p("in.txt"), p("c.json"), p("pub.json"))
        return decrypt_file(p("c.json"), p("out.txt"), p("priv.json"))


samples = {
    "пустой файл": "",
    "один символ": "a",
    "латиница": "The quick brown fox jumps over the lazy dog.",
    "кириллица": "Съешь же ещё этих мягких французских булок, да выпей чаю.",
    "переносы строк и табы": "строка 1\r\nстрока 2\n\tс табуляцией\n\n",
    "эмодзи и спецсимволы": "Ключ 🔑 → шифр 🔒 ≠ текст ✓ №4",
    "длинный текст": "Merkle-Hellman " * 500,
}

for name, text in samples.items():
    for n, t in ((8, 1), (12, 2), (16, 3), (32, 5)):     # в т.ч. n, не кратное 8
        assert roundtrip(text, n, t, seed=hash((name, n, t)) % 10**6) == text, (name, n, t)
    print(f"OK: {name}")

# --- ключи, прочитанные из файла, совпадают с исходными ---
pub0, priv0 = generate_keys(16, 4, random.Random(99))
with tempfile.TemporaryDirectory() as d:
    save_keys(pub0, priv0, os.path.join(d, "a.json"), os.path.join(d, "b.json"))
    assert load_public_key(os.path.join(d, "a.json")) == pub0
    assert load_private_key(os.path.join(d, "b.json")) == priv0
print("OK: сохранение/загрузка ключей")

# --- неверный закрытый ключ не расшифровывает файл ---
with tempfile.TemporaryDirectory() as d:
    p = lambda name: os.path.join(d, name)
    pubA, privA = generate_keys(16, 3, random.Random(1))
    pubB, privB = generate_keys(16, 3, random.Random(2))
    save_keys(pubA, privA, p("pubA.json"), p("privA.json"))
    save_keys(pubB, privB, p("pubB.json"), p("privB.json"))
    secret = "Секретное сообщение, которое нельзя прочитать чужим ключом."
    with open(p("in.txt"), "w", encoding="utf-8") as f:
        f.write(secret)
    encrypt_file(p("in.txt"), p("c.json"), p("pubA.json"))
    try:
        wrong = decrypt_file(p("c.json"), p("out.txt"), p("privB.json"))
        assert wrong != secret
    except (ValueError, UnicodeDecodeError):
        pass                                   # ожидаемый отказ при чужом ключе
    assert decrypt_file(p("c.json"), p("out.txt"), p("privA.json")) == secret
print("OK: чужой ключ не подходит, свой — подходит")

# --- несовпадение размера блока ---
with tempfile.TemporaryDirectory() as d:
    p = lambda name: os.path.join(d, name)
    pub16, priv16 = generate_keys(16, 2, random.Random(5))
    pub8, priv8 = generate_keys(8, 2, random.Random(6))
    save_keys(pub16, priv16, p("pub16.json"), p("priv16.json"))
    save_keys(pub8, priv8, p("pub8.json"), p("priv8.json"))
    with open(p("in.txt"), "w", encoding="utf-8") as f:
        f.write("тест")
    encrypt_file(p("in.txt"), p("c.json"), p("pub16.json"))
    try:
        decrypt_file(p("c.json"), p("out.txt"), p("priv8.json"))
        raise AssertionError("ожидалась ошибка")
    except ValueError:
        pass
print("OK: контроль размера блока")
print("\nВсе файловые тесты пройдены.")

OK: пустой файл
OK: один символ
OK: латиница
OK: кириллица
OK: переносы строк и табы
OK: эмодзи и спецсимволы
OK: длинный текст
OK: сохранение/загрузка ключей
OK: чужой ключ не подходит, свой — подходит
OK: контроль размера блока

Все файловые тесты пройдены.


## 10. Выводы

* Реализована многократная схема Merkle-Hellman: генерация ключей с $t$ последовательными модульными умножениями, шифрование суммой подмножества открытого ключа и дешифрование последовательным снятием умножений ($d_j = W_j^{-1} d_{j+1} \bmod M_j$) с решением лёгкой сверхвозрастающей задачи.
* Корректность следует из условия $M_j > \sum a_i^{(j-1)}$: на каждом шаге сумма выбранных элементов не превосходит модуль, поэтому приведение по модулю не искажает её.
* Расшифровка проверена на примере из методички (при $t=1$ значения совпали: $c = 1605$, $d = 136$) и на тысячах случайных ключей.
* Ключи и шифротекст сохраняются в JSON-файлы; дешифрование файла выполняется только по закрытому ключу и шифротексту, загруженным с диска. Проверены пустой файл, юникод, эмодзи, переносы строк, длинный текст, размеры блока, не кратные 8, и отказ при чужом ключе.
* Обратные элементы $W_j^{-1}$ вычисляются расширенным алгоритмом Евклида (2.107).
* Замечание по безопасности: хотя задача о рюкзаке NP-полна в общем случае, схема Merkle-Hellman (и её многократный вариант) на практике взломана — базовая схема атакой Shamir (1982), многократная — атаками на основе решёточной редукции (Brickell, 1985). Поэтому схема представляет в основном исторический и учебный интерес.